# 06 · Loop engineering

Oltre all'agente base esistono altri "loop" che lo circondano
([The Art of Loop Engineering](https://www.langchain.com/blog/the-art-of-loop-engineering)).
Ne vediamo tre, ognuno costruito da zero e commentato:
1. **verifica** con una rubrica (Loop 2);
2. **trigger a eventi** (Loop 3);
3. **hill climbing**: dai dati d'uso a una proposta di miglioramento (Loop 4).

## Obiettivi, prerequisiti e modalità di lettura

Studierai verification, trigger e improvement loop. Durata: 35–45 minuti. Proposta non significa applicazione: il gate resta separato.

Ogni blocco di codice è preceduto da una spiegazione e seguito da un **output
atteso**. Quando interviene un modello, l'output atteso descrive proprietà e
invarianti, non una frase letterale. Esegui le celle in ordine e non saltare i
casi negativi: mostrano il confine del meccanismo, non un incidente del corso.

## Setup (autonomo)

Ogni notebook è **indipendente**: non importa nulla dal progetto. Qui carichiamo la chiave
API dal file `.env` e creiamo un modello. Esegui le celle in ordine dall'alto verso il basso.

### Spiegazione del blocco · Setup

Configurazione locale e chiave vengono validate prima dei quattro loop.

In [ ]:
# Carichiamo le variabili d'ambiente dal file `.env`.
# Lo cerchiamo nella cartella corrente e in quelle superiori, così il notebook
# funziona sia se avviato dalla radice del progetto sia dalla cartella `notebooks`.
import os
from pathlib import Path

from dotenv import load_dotenv


def trova_env() -> Path:
    for cartella in (Path.cwd(), *Path.cwd().resolve().parents):
        if (cartella / ".env").is_file():
            return cartella / ".env"
    raise FileNotFoundError("File .env non trovato: copia .env.example in .env e aggiungi la chiave.")


env_file = trova_env()
load_dotenv(env_file, override=False)          # carica le variabili senza sovrascrivere quelle già presenti
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY mancante nel file .env"
print("Ambiente caricato da:", env_file)

### Output atteso

Percorso `.env` caricato.

### Spiegazione del blocco · Modello

Un solo client alimenta agente, giudice e analista. In produzione questi ruoli possono usare tier diversi.

In [ ]:
# `ChatOpenAI` è il wrapper LangChain attorno al modello.
# Lo creiamo una volta e lo riusiamo in tutto il notebook.
import warnings

from langchain_openai import ChatOpenAI

# Il client OpenAI (Responses API) dichiara `output` come unione di ~30 tipi;
# pydantic avvisa per tutti gli altri ad ogni structured output. Rumore innocuo.
warnings.filterwarnings(
    "ignore", message="Pydantic serializer warnings", category=UserWarning
)

MODELLO = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")   # modello economico, va bene per imparare
model = ChatOpenAI(
    model=MODELLO,
    use_responses_api=True,   # API "responses" di OpenAI
    store=False,              # non conservare la conversazione sui server OpenAI
)
print("Modello pronto:", MODELLO)

### Output atteso

Nome del modello pronto.

## Loop 1 · L'agente base

Il punto di partenza: un agente qualsiasi. Sarà l'oggetto che gli altri loop "circondano".

### Spiegazione del blocco · Loop agente di base

`esegui` nasconde il boilerplate del graph e restituisce solo testo finale. Gli altri loop chiameranno questa funzione.

In [ ]:
from langchain.agents import create_agent

agente = create_agent(model=model, tools=[], system_prompt="Rispondi in modo utile e conciso.")


def esegui(domanda: str) -> str:
    esito = agente.invoke({"messages": [{"role": "user", "content": domanda}]})
    return esito["messages"][-1].text

### Output atteso

Nessun output. Agente e helper sono definiti.

## Loop 2 · Verifica con una rubrica

Un secondo modello fa da **giudice**: dà un voto 0–1 a più criteri e un feedback.
La soglia (passa/non passa) la decidiamo noi in Python — deterministica, non dal modello.

Nota: con lo *structured output* di OpenAI lo schema deve essere **chiuso** (campi espliciti),
quindi elenchiamo i criteri come campi, non come dizionario libero.

### Spiegazione del blocco · Schema della rubrica

Pydantic vincola punteggi tra zero e uno e rende il feedback strutturato. Uno schema chiuso riduce risposte del giudice difficili da interpretare.

In [ ]:
# Lo schema del giudizio: un voto per criterio + un feedback. Campi espliciti = schema chiuso.
from pydantic import BaseModel, Field


class Giudizio(BaseModel):
    completezza: float = Field(ge=0, le=1)
    chiarezza: float = Field(ge=0, le=1)
    feedback: str = ""

### Output atteso

Nessun output. La classe `Giudizio` è disponibile.

### Spiegazione del blocco · Giudice strutturato

Il modello viene adattato a restituire `Giudizio`. La soglia rimane codice deterministico, separata dalla valutazione probabilistica.

In [ ]:
# Il giudice è il modello con output strutturato sullo schema Giudizio.
giudice = model.with_structured_output(Giudizio)


def valuta(domanda: str, risposta: str, soglia: float = 0.7):
    g = giudice.invoke([
        {"role": "system", "content": "Valuta la RISPOSTA rispetto alla DOMANDA, 0-1 per criterio."},
        {"role": "user", "content": f"DOMANDA: {domanda}\nRISPOSTA: {risposta}"},
    ])
    voto = (g.completezza + g.chiarezza) / 2      # media dei criteri
    return voto >= soglia, voto, g.feedback        # passa?, voto, feedback

### Output atteso

Nessun output. `valuta` restituirà `(passa, voto, feedback)`.

### Spiegazione del blocco · Verification loop

Prima si produce una risposta, poi la rubrica la valuta. Se non passa, il feedback viene incorporato in un nuovo tentativo anziché ripetere identica richiesta.

In [ ]:
# Ciclo di verifica: se non passa, rimandiamo indietro il feedback e riproviamo.
domanda = "Spiega cos'è un checkpoint in LangGraph."
risposta = esegui(domanda)
passa, voto, feedback = valuta(domanda, risposta)
print("voto:", round(voto, 2), "passa:", passa)

if not passa:
    risposta = esegui(f"{domanda}\nMigliora tenendo conto di questo feedback: {feedback}")
    print("--- risposta migliorata ---")
print(risposta)

### Output atteso

Voto tra `0` e `1`, booleano `passa` e risposta finale. Può comparire una seconda risposta migliorata.

## Loop 3 · Trigger a eventi

Finora l'agente parte perché lo chiamiamo noi. Un **trigger** lo avvia al verificarsi di un
evento: un orario (cron) o un webhook. Costruiamo un mini valutatore di espressioni cron.

### Spiegazione del blocco · Parser cron minimale

Le funzioni implementano wildcard, liste e intervalli `*/n` per minuto e ora. Il sottoinsieme è sufficiente a spiegare il trigger senza dipendere da scheduler esterni.

In [ ]:
from datetime import datetime, timezone


def campo_cron(campo: str, valore: int) -> bool:
    # Supporta '*', liste '1,2', e step '*/5'. Sufficiente per capire l'idea.
    if campo == "*":
        return True
    if campo.startswith("*/"):
        return valore % int(campo[2:]) == 0
    return valore in {int(x) for x in campo.split(",")}


def cron_combacia(espressione: str, momento: datetime) -> bool:
    minuto, ora = espressione.split()[:2]     # usiamo i primi due campi: minuto e ora
    return campo_cron(minuto, momento.minute) and campo_cron(ora, momento.hour)

### Output atteso

Nessun output. `cron_combacia` è pronta per timestamp timezone-aware.

### Spiegazione del blocco · Event-driven loop

L'espressione ogni minuto combacia sempre, quindi simula uno scheduler che avvia l'agente automaticamente.

In [ ]:
# Uno "scheduler" minimale: se l'espressione combacia con l'ora attuale, avvia l'agente.
adesso = datetime.now(timezone.utc)
if cron_combacia("* * * * *", adesso):        # '* * * * *' = ogni minuto -> combacia sempre
    print(esegui("Scrivi una frase che conferma l'avvio automatico."))

### Output atteso

Una frase generata che conferma l'avvio automatico.

## Loop 4 · Hill climbing

L'idea più avanzata: usare i **dati d'uso** per migliorare la configurazione dell'agente.
Un agente d'analisi legge un piccolo report e propone modifiche, entro una lista sicura di
campi. **Propose-only**: la proposta va poi rivista da un umano prima di applicarla.

### Spiegazione del blocco · Schema della proposta

La whitelist strutturale limita ciò che il loop di miglioramento può cambiare. Campi fuori schema non diventano configurazione attiva.

In [ ]:
# La proposta ha SOLO campi consentiti (whitelist strutturale) -> schema chiuso, override sicuri.
class Proposta(BaseModel):
    sintesi: str = ""
    aggiunta_al_prompt: str | None = None       # testo da aggiungere al system prompt
    max_chiamate_tool: int | None = None        # nuovo limite di tool call

### Output atteso

Nessun output. `Proposta` consente solo sintesi, addendum e limite tool.

### Spiegazione del blocco · Analisi propose-only

L'analista legge un report aggregato e produce una proposta, ma il blocco non la applica. Eval, canary e approvazione vengono prima di ogni promozione reale.

In [ ]:
analista = model.with_structured_output(Proposta)

# Un finto report dei run recenti (in un sistema vero verrebbe dai log/trace).
report = "Run totali: 8\nRun falliti: 3\nErrore ricorrente: retry ripetuti su web_read"

proposta = analista.invoke([
    {"role": "system", "content": "Analizza il REPORT e proponi migliorie solo se giustificate."},
    {"role": "user", "content": report},
])
print("Sintesi:", proposta.sintesi)
print("Aggiunta al prompt:", proposta.aggiunta_al_prompt)
print("Nuovo limite tool:", proposta.max_chiamate_tool)

### Output atteso

Sintesi e due override opzionali. Valori possono essere `None` se il modello non trova evidenza sufficiente.

## Prova tu

- Nel Loop 2, abbassa la soglia a 0.9 e osserva quante iterazioni servono.
- Nel Loop 4, applica davvero la proposta ricreando l'agente con `system_prompt` aggiornato —
  ma solo dopo averla letta: è il principio *propose-only + revisione umana*.

**Idea chiave**: l'agente è il cuore; verifica, eventi e auto-miglioramento sono i loop che
lo rendono affidabile, autonomo e capace di migliorare nel tempo.

## Laboratorio aggiuntivo

Gli esempi seguenti riusano quanto costruito sopra. Il primo amplia il caso normale; il
secondo esercita un confine, un errore o una proprietà che spesso causa bug reali.

## Esempio aggiuntivo: matrice di trigger cron

### Spiegazione del blocco

Una tabella di casi rende evidente la differenza tra wildcard, step e ora specifica.

In [ ]:
momento = datetime(2026, 7, 15, 10, 30, tzinfo=timezone.utc)
for espressione in ("* * * * *", "*/15 * * * *", "0 10 * * *", "30 10 * * *"):
    print(espressione, "->", cron_combacia(espressione, momento))

### Output atteso

Risultati: `True`, `True`, `False`, `True` per il timestamp delle 10:30.

## Esempio aggiuntivo: gate deterministico per una proposta

### Spiegazione del blocco

La proposta del modello non viene applicata direttamente. Un gate controlla limiti e campi consentiti.

In [ ]:
def proposta_ammissibile(valore: Proposta) -> tuple[bool, list[str]]:
    problemi = []
    if valore.max_chiamate_tool is not None and not 1 <= valore.max_chiamate_tool <= 50:
        problemi.append("max_chiamate_tool fuori intervallo")
    if valore.aggiunta_al_prompt and len(valore.aggiunta_al_prompt) > 500:
        problemi.append("aggiunta_al_prompt troppo lunga")
    return not problemi, problemi

print(proposta_ammissibile(Proposta(max_chiamate_tool=20)))
print(proposta_ammissibile(Proposta(max_chiamate_tool=500)))

### Output atteso

Prima proposta ammessa; seconda respinta con problema sul limite tool.

## Riepilogo e troubleshooting

Prima di proseguire, prova a spiegare con parole tue: quale stato è cambiato, quale
componente ha preso la decisione e quale prova rende osservabile l'esito.

Se una cella fallisce:

1. rileggi l'output atteso e individua la prima invariante non rispettata;
2. verifica di aver eseguito tutte le celle precedenti nello stesso kernel;
3. per i notebook live, controlla `.env`, modello disponibile e quota API;
4. riavvia il kernel solo dopo aver conservato eventuali file che vuoi ispezionare;
5. non correggere un caso negativo: l'errore previsto è parte dell'esempio.